# 🎵 Free AI Song Studio: ACE-Step 1.5 on Google Colab (free T4)

Make full songs **with vocals** from your lyrics and a style description, for free.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom.

| Step | What it does | Time |
|---|---|---|
| 1 | Settings + GPU check (+ Google Drive for saving) | seconds |
| 2 | Install ACE-Step 1.5 and download the models | ~5–10 min per session |
| 3 | Launch the studio: you get a link to a web app | ~3–8 min |
| 4 | Save and master your songs (loud, polished, release-ready) | ~1 min per song |

**Why this model:** [ACE-Step 1.5](https://github.com/ace-step/ACE-Step-1.5) is the strongest open song model that fits a free T4 (15 GB GPU memory, no BF16, ~12 GB system RAM).
Its code *and* weights are MIT-licensed, and the model card allows commercial use of the music you generate.
Models that score closer to Suno (YuE2, HeartMuLa, LeVo 2) need a 24 GB BF16 GPU or more RAM than free Colab gives you.

In [ ]:
#@title 1️⃣ Settings + GPU check
#@markdown **Model.** `turbo` is the fastest and has the best quality rating from the ACE-Step team. `sft` is slower (50 steps) and gives you more control.
DIT_MODEL = "acestep-v15-turbo"  #@param ["acestep-v15-turbo", "acestep-v15-sft"]
#@markdown **Song planner (language model).** 1.7B writes better song structure. Pick 0.6B only if you get "out of memory" errors.
LM_MODEL = "acestep-5Hz-lm-1.7B"  #@param ["acestep-5Hz-lm-1.7B", "acestep-5Hz-lm-0.6B"]
#@markdown **Planner backend.** Leave `vllm` unless step 3 fails with planner/LM errors, then switch to `pt`.
LM_BACKEND = "vllm"  #@param ["vllm", "pt"]
#@markdown **Save finished songs to Google Drive** (so they survive when Colab disconnects).
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "AI-Music"  #@param {type:"string"}
#@markdown **ACE-Step version.** `pinned` is the version this notebook was written against. `latest` picks up new features but may change behavior.
ACESTEP_VERSION = "pinned"  #@param ["pinned", "latest"]

import os
import subprocess

REPO_DIR = "/content/ACE-Step-1.5"
PINNED_COMMIT = "ca1e85fe9430179831e6bc6be790c332190a3866"  # ACE-Step-1.5 main, 2026-08-29

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode != 0:
    raise RuntimeError(
        "No GPU attached. Go to Runtime → Change runtime type → T4 GPU, then run this cell again."
    )
print("GPU:", gpu.stdout.strip())

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
else:
    OUT_DIR = "/content/songs"
os.makedirs(OUT_DIR, exist_ok=True)
print("Finished songs will be saved to:", OUT_DIR)

In [ ]:
#@title 2️⃣ Install ACE-Step 1.5 + download models (~5–10 min)
import os
import subprocess


def sh(cmd, cwd=None):
    print(f"$ {cmd}", flush=True)
    env = dict(os.environ, UV_NO_PROGRESS="1", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed: {cmd}")


if not os.path.isdir(f"{REPO_DIR}/.git"):
    sh(f"git clone https://github.com/ace-step/ACE-Step-1.5.git {REPO_DIR}")
if ACESTEP_VERSION == "pinned":
    sh(f"git fetch --quiet origin {PINNED_COMMIT} && git checkout --quiet {PINNED_COMMIT}", cwd=REPO_DIR)
else:
    sh("git fetch --quiet origin main && git checkout --quiet origin/main", cwd=REPO_DIR)

# uv builds an isolated Python 3.12 environment with the exact versions ACE-Step was tested with,
# so nothing clashes with Colab's own packages.
sh("pip install -q uv")
sh("uv sync", cwd=REPO_DIR)

# The main bundle holds the turbo song model, the 1.7B planner, the audio decoder (VAE) and the text encoder.
sh("uv run acestep-download", cwd=REPO_DIR)
if DIT_MODEL != "acestep-v15-turbo":
    sh(f"uv run acestep-download --model {DIT_MODEL}", cwd=REPO_DIR)
if LM_MODEL != "acestep-5Hz-lm-1.7B":
    sh(f"uv run acestep-download --model {LM_MODEL}", cwd=REPO_DIR)

print("\n✅ Installed. Run step 3.")

In [ ]:
#@title 3️⃣ Launch the song studio (prints a link to open)
import os
import re
import subprocess
import time

LOG_PATH = "/content/acestep_studio.log"
PORT = 7860

# Stop a studio left over from an earlier run so it doesn't hold GPU memory.
if "studio_proc" in globals() and studio_proc.poll() is None:
    studio_proc.terminate()
    studio_proc.wait(timeout=60)
subprocess.run(["pkill", "-f", f"{REPO_DIR}/.venv/bin/acestep"], check=False)
time.sleep(2)

cmd = [
    f"{REPO_DIR}/.venv/bin/acestep",
    "--share",
    "--server-name", "0.0.0.0",
    "--port", str(PORT),
    "--init_service", "true",
    "--config_path", DIT_MODEL,
    "--lm_model_path", LM_MODEL,
    "--backend", LM_BACKEND,
]
log_file = open(LOG_PATH, "w")
studio_proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, stdout=log_file, stderr=subprocess.STDOUT,
    env=dict(os.environ, PYTHONUNBUFFERED="1"),
)
print("Starting the studio. The first launch loads and optimizes the models (3–8 min)...", flush=True)

public_url, local_ready = None, False
deadline = time.time() + 25 * 60
last_note = time.time()
while time.time() < deadline:
    time.sleep(5)
    with open(LOG_PATH, errors="ignore") as f:
        log_text = f.read()
    m = re.search(r"https://[a-z0-9-]+\.gradio\.live", log_text)
    if m:
        public_url = m.group(0)
        break
    local_ready = local_ready or "Running on local URL" in log_text
    if studio_proc.poll() is not None:
        break
    if local_ready and "Could not create share link" in log_text:
        break
    if time.time() - last_note > 30:
        lines = [l for l in log_text.splitlines() if l.strip()]
        print("  …", lines[-1][:160] if lines else "starting", flush=True)
        last_note = time.time()

if public_url:
    print(f"\n✅ Studio ready: {public_url}")
    print("   Works on phone and desktop. Keep this Colab tab open while you use it.")
elif local_ready and studio_proc.poll() is None:
    from google.colab.output import eval_js
    proxy_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print(f"\n✅ Studio ready (public link unavailable, using Colab's private link): {proxy_url}")
    print("   This link only works in this browser while this notebook is open.")
else:
    print("\n".join(open(LOG_PATH, errors="ignore").read().splitlines()[-60:]))
    raise RuntimeError(
        "The studio did not start. See the log above. Common fixes: set LM_BACKEND to 'pt' or "
        "LM_MODEL to the 0.6B planner in step 1, run step 1 again, then run this cell again."
    )

## 🎚️ How to get Suno-like results

The studio opens in **Text2Music**. Fill in two boxes:

**1. Caption (the style).** Write it the way a producer would describe the track: genre and sub-genre, tempo feel, instruments, vocal type, mood and production. For example:

> *Modern synth-pop at 118 BPM with a four-on-the-floor beat, warm analog bass, shimmering arpeggiated synths and gated reverb snares. Breathy, intimate female lead vocal with stacked harmonies in the chorus. Bittersweet and nostalgic, with a polished, wide, radio-ready mix.*

Avoid vague captions ("good music") and contradictions ("fast slow"). Don't use artist names.

**2. Lyrics, with structure tags.** Tags tell the model where each section goes:

```
[Intro]

[Verse 1]
City lights are fading on the train ride home
Your voice still in my headphones on a dying phone

[Pre-Chorus]
And I know, and I know

[Chorus]
We were neon in the dark, we were louder than the rain
...

[Bridge]
...

[Outro]
```

Instrument cues inside tags also work, for example `[Instrumental Break - Synth Solo]`. Keep lines short and singable, and use one language per song.

**3. Habits that improve quality the most:**
1. **Generate several takes and keep the best.** Set the batch size to 2–4 (up to 4 fits on a T4) and generate a few rounds. Picking from more takes is the single biggest lever: it's how YuE2 beat Suno v5 in its own benchmark.
2. **Leave "thinking" on, with the 1.7B planner.** It plans BPM, key and song structure before the audio is made.
3. **Fix weak sections with Repaint** instead of starting over. Give it the start and end seconds of the bad part, and it regenerates just that part.
4. **Master the final take** in step 4.

When you're done generating, run step 4. It finds the songs the studio saved.

In [ ]:
#@title 4️⃣ Save + master your songs
#@markdown **Reference matching** makes your song match the loudness and tone of a professional track you upload (pick one in a similar genre).
#@markdown **Loudness only** sets a streaming-standard -14 LUFS level and needs no upload.
MASTERING = "Match a reference song (best)"  #@param ["Match a reference song (best)", "Loudness only (-14 LUFS)", "No mastering (copy as-is)"]
WHICH_SONGS = "Newest batch"  #@param ["Newest batch", "All songs from this session"]

import glob
import json
import os
import shutil
import subprocess

from IPython.display import Audio, display

AUDIO_EXT = (".flac", ".wav", ".mp3", ".opus", ".aac", ".m4a", ".ogg")
RESULTS_ROOT = os.path.join(REPO_DIR, "gradio_outputs")


def find_songs(results_root, newest_only):
    batches = sorted(
        (d for d in glob.glob(os.path.join(results_root, "*")) if os.path.isdir(d)),
        key=os.path.getmtime,
    )
    chosen = batches[-1:] if newest_only else batches
    return sorted(
        p for d in chosen
        for p in glob.glob(os.path.join(d, "**", "*"), recursive=True)
        if p.lower().endswith(AUDIO_EXT)
    )


def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{cmd[0]} failed:\n{r.stderr[-2000:]}")
    return r


def to_wav(src, dst, rate=44100):
    run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ar", str(rate), "-ac", "2", "-c:a", "pcm_s24le", dst])


def to_mp3(src, dst):
    run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-c:a", "libmp3lame", "-b:a", "320k", dst])


def loudnorm(src, dst, target_lufs=-14.0, true_peak=-1.0, lra=11.0):
    # Two-pass EBU R128 loudness normalization: measure first, then apply a linear gain.
    base = f"loudnorm=I={target_lufs}:TP={true_peak}:LRA={lra}"
    probe = subprocess.run(
        ["ffmpeg", "-hide_banner", "-i", src, "-af", f"{base}:print_format=json", "-f", "null", "-"],
        capture_output=True, text=True,
    )
    m = json.loads(probe.stderr[probe.stderr.rindex("{"):])
    af = (f"{base}:measured_I={m['input_i']}:measured_TP={m['input_tp']}:measured_LRA={m['input_lra']}"
          f":measured_thresh={m['input_thresh']}:offset={m['target_offset']}:linear=true")
    run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-af", af, "-ar", "48000", "-c:a", "pcm_s24le", dst])


def master_song(song, mode, reference_wav, out_dir, work_dir):
    stem = f"{os.path.basename(os.path.dirname(song))}_{os.path.splitext(os.path.basename(song))[0]}"
    final_wav = os.path.join(out_dir, f"{stem}_master.wav")
    if mode.startswith("Match"):
        import matchering as mg
        target_wav = os.path.join(work_dir, f"{stem}_in.wav")
        to_wav(song, target_wav)
        mg.process(target=target_wav, reference=reference_wav, results=[mg.pcm24(final_wav)])
    elif mode.startswith("Loudness"):
        loudnorm(song, final_wav)
    else:
        final_wav = os.path.join(out_dir, stem + os.path.splitext(song)[1])
        shutil.copy2(song, final_wav)
    final_mp3 = os.path.splitext(final_wav)[0] + ".mp3"
    to_mp3(final_wav, final_mp3)
    return final_wav, final_mp3


songs = find_songs(RESULTS_ROOT, WHICH_SONGS == "Newest batch")
if not songs:
    raise RuntimeError(f"No songs found in {RESULTS_ROOT}. Generate something in the studio first (step 3).")
print(f"Found {len(songs)} song(s).")

work_dir = "/content/mastering_tmp"
os.makedirs(work_dir, exist_ok=True)
reference_wav = None
if MASTERING.startswith("Match"):
    subprocess.run(["pip", "install", "-q", "matchering"], check=True)
    from google.colab import files
    print("Upload a professionally mastered song in a similar genre (mp3/wav/flac):")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No reference uploaded. Run again and upload one, or choose 'Loudness only'.")
    reference_wav = os.path.join(work_dir, "reference.wav")
    to_wav(os.path.abspath(next(iter(uploaded))), reference_wav)

for song in songs:
    wav, mp3 = master_song(song, MASTERING, reference_wav, OUT_DIR, work_dir)
    print(f"✅ {os.path.basename(mp3)}  +  {os.path.basename(wav)}  →  {OUT_DIR}")
    display(Audio(filename=mp3))

## 🛠️ Troubleshooting

- **"CUDA out of memory"**: in the studio, lower the batch size or duration. Or pick the 0.6B planner in step 1, then run steps 1 and 3 again.
- **Step 3 fails with LM/planner errors**: set `LM_BACKEND` to `pt` in step 1, then run steps 1 and 3 again.
- **The `gradio.live` link stops working**: run step 3 again for a fresh link. If no public link can be created, the cell prints a private Colab link instead.
- **Colab disconnected**: free sessions end after idling or after a few hours. Files on the Colab machine are wiped; anything saved to Google Drive in step 4 is kept. Run steps 1–3 again to continue.
- **"Cannot connect to GPU backend"**: you've hit Colab's free GPU limit for now. Try again later. [Kaggle Notebooks](https://www.kaggle.com/code) also offer free T4 GPUs with a weekly quota.

**License:** ACE-Step 1.5 code and weights are MIT. The [model card](https://huggingface.co/ACE-Step/Ace-Step1.5) says the music you generate can be used commercially. Use only lyrics you wrote or have rights to.